# Cross-Encoder Reranking

Bi-encoder (dense retrieval) models embed the query and each document independently, then compare them by cosine similarity. This is fast — similarity is a single dot product — but it approximates relevance, because the query and document never interact during encoding. Cross-encoders take a different approach: they score a (query, document) pair jointly, allowing the model to capture token-level interactions between the question and the passage. This produces far more precise relevance scores, but at a cost: cross-encoders cannot be pre-computed, so they must be run at query time for every candidate document.

The standard pattern is to **retrieve a large candidate set cheaply with a bi-encoder** (k=20 or more) and then **rerank to the final top-k using a cross-encoder**.

**What you'll build:** A two-stage retrieval pipeline that retrieves 20 candidate documents using dense vector search and reranks them to the top 5 using a cross-encoder, then passes those 5 to the LLM.

**Time:** ~15 min | **Difficulty:** Intermediate

## Prerequisites

- Completed the [Basic RAG](https://synapsekit.github.io/synapsekit-docs/guides/rag/) guide
- `sentence-transformers` for the local cross-encoder model
- `OPENAI_API_KEY` set in your environment

## What you'll learn

- Why two-stage retrieval (retrieve wide, rerank narrow) outperforms single-stage retrieval
- How `CrossEncoderReranker` scores query-document pairs
- How to configure the candidate pool size and final top-k independently
- How to swap in an API-based reranker (Cohere, Jina) instead of a local model

In [ ]:
!pip install synapsekit sentence-transformers -q

## Environment Setup

In [ ]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-..."  # set your key here or via environment
# os.environ["COHERE_API_KEY"] = "..."     # only needed if using CohereReranker

## Step 1: Install and configure

In [ ]:
import asyncio
import os

from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.rerankers import CrossEncoderReranker
from synapsekit.retrievers import VectorRetriever

llm = OpenAILLM(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = InMemoryVectorStore(embeddings=embeddings)

## Step 2: Load documents

In [ ]:
docs = [
    "Python's asyncio library provides event loop-based concurrency for I/O-bound tasks.",
    "The Global Interpreter Lock (GIL) in CPython prevents true parallelism for CPU-bound code.",
    "Threading in Python is suitable for I/O-bound tasks but not CPU-bound tasks due to the GIL.",
    "Multiprocessing bypasses the GIL by running each process in a separate Python interpreter.",
    "async/await syntax in Python 3.5+ enables cooperative multitasking without threads.",
    "Celery is a distributed task queue for Python that supports asynchronous job execution.",
    "Ray is a framework for distributed computing in Python, optimised for ML workloads.",
    "concurrent.futures provides a high-level interface for both thread and process pools.",
    "The asyncio.gather() function runs multiple coroutines concurrently in the same event loop.",
    "Gevent uses greenlets to provide lightweight concurrency in Python without native threads.",
    "WSGI is a synchronous interface for Python web servers and does not support async handlers.",
    "ASGI extends WSGI to support asynchronous Python web frameworks like FastAPI and Starlette.",
    "uvloop is a fast asyncio event loop implementation built on top of libuv.",
    "trio is an alternative async library for Python with a structured concurrency model.",
    "Python's queue.Queue is thread-safe and suitable for producer-consumer patterns with threads.",
    "asyncio.Queue provides a thread-unsafe but coroutine-safe queue for async workflows.",
    "Process pools in Python use pickle to serialise arguments, which can be a bottleneck.",
    "Dask provides parallel computing for Python with lazy evaluation and task graph scheduling.",
    "Numba compiles Python functions to native machine code using LLVM, bypassing the GIL.",
    "The multiprocessing.Manager class enables shared state between processes via a proxy server.",
]

await vectorstore.aadd(docs)
print(f"Added {len(docs)} documents to the vector store.")

## Step 3: Set up the two-stage retrieval pipeline

The key numbers here are `candidates` (how many documents the bi-encoder retrieves) and `top_k` (how many the cross-encoder keeps). A candidate pool of 20 gives the cross-encoder enough to work with. Final `top_k=5` keeps the LLM's context tight and reduces generation cost.

In [ ]:
# Stage 1: retrieve a large candidate pool using fast bi-encoder similarity
base_retriever = VectorRetriever(
    vectorstore=vectorstore,
    top_k=20,  # retrieve wide — precision does not matter here, recall does
)

# Stage 2: score each candidate against the query jointly using a cross-encoder
reranker = CrossEncoderReranker(
    model="cross-encoder/ms-marco-MiniLM-L-6-v2",  # local model, no API key required
    top_k=5,  # keep only the 5 most precisely relevant documents
)

print("Two-stage retrieval pipeline configured.")
print("  Stage 1: VectorRetriever top_k=20")
print("  Stage 2: CrossEncoderReranker top_k=5")

## Step 4: Retrieve candidates and rerank

In [ ]:
async def reranked_retrieve():
    query = "What is the best way to run CPU-bound Python code in parallel?"

    # Stage 1: broad retrieval
    candidates = await base_retriever.aretrieve(query)
    print(f"Stage 1 candidates: {len(candidates)}")
    for doc, score in candidates[:3]:
        print(f"  [{score:.3f}] {doc[:70]}")

    print()

    # Stage 2: cross-encoder reranking
    reranked = await reranker.arerank(query, candidates)
    print(f"Stage 2 top-{reranker.top_k} after reranking:")
    for doc, score in reranked:
        print(f"  [{score:.4f}] {doc[:70]}")

asyncio.run(reranked_retrieve())

## Step 5: Wire both stages into the RAG pipeline

`CrossEncoderReranker` can be passed directly to `RAG` as a `reranker` parameter. The pipeline handles the two-stage flow internally.

In [ ]:
rag = RAG(
    llm=llm,
    retriever=base_retriever,
    reranker=reranker,
)
print("RAG pipeline assembled with cross-encoder reranking.")

## Step 6: Ask a question and compare with and without reranking

In [ ]:
async def compare_pipelines():
    query = "What is the best way to run CPU-bound Python code in parallel?"

    # Without reranking
    rag_plain = RAG(llm=llm, retriever=VectorRetriever(vectorstore=vectorstore, top_k=5))
    result_plain = await rag_plain.aquery(query)

    # With reranking
    result_reranked = await rag.aquery(query)

    print("Without reranking:")
    print(result_plain.answer[:300])
    print("\nWith reranking:")
    print(result_reranked.answer[:300])

asyncio.run(compare_pipelines())

## Step 7: Use an API-based reranker

For production deployments, an API-based reranker like Cohere or Jina avoids loading a local model and can be more accurate.

In [ ]:
# Uncomment and set COHERE_API_KEY to use the Cohere reranker
# from synapsekit.rerankers import CohereReranker
#
# reranker_cohere = CohereReranker(
#     api_key=os.environ["COHERE_API_KEY"],
#     model="rerank-english-v3.0",
#     top_k=5,
# )
#
# rag_cohere = RAG(llm=llm, retriever=base_retriever, reranker=reranker_cohere)

print("API-based reranker example shown above (commented out — requires COHERE_API_KEY).")

## Complete Working Example

A single self-contained cell that runs the entire cross-encoder reranking pipeline end-to-end.

In [ ]:
import asyncio
import os

from synapsekit import RAG
from synapsekit.llms.openai import OpenAILLM
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.memory import InMemoryVectorStore
from synapsekit.rerankers import CrossEncoderReranker
from synapsekit.retrievers import VectorRetriever

DOCS = [
    "Python's asyncio library provides event loop-based concurrency for I/O-bound tasks.",
    "The Global Interpreter Lock (GIL) in CPython prevents true parallelism for CPU-bound code.",
    "Threading in Python is suitable for I/O-bound tasks but not CPU-bound tasks due to the GIL.",
    "Multiprocessing bypasses the GIL by running each process in a separate Python interpreter.",
    "async/await syntax in Python 3.5+ enables cooperative multitasking without threads.",
    "Celery is a distributed task queue for Python that supports asynchronous job execution.",
    "Ray is a framework for distributed computing in Python, optimised for ML workloads.",
    "concurrent.futures provides a high-level interface for both thread and process pools.",
    "The asyncio.gather() function runs multiple coroutines concurrently in the same event loop.",
    "Gevent uses greenlets to provide lightweight concurrency in Python without native threads.",
    "WSGI is a synchronous interface for Python web servers and does not support async handlers.",
    "ASGI extends WSGI to support asynchronous Python web frameworks like FastAPI and Starlette.",
    "uvloop is a fast asyncio event loop implementation built on top of libuv.",
    "trio is an alternative async library for Python with a structured concurrency model.",
    "Python's queue.Queue is thread-safe and suitable for producer-consumer patterns with threads.",
    "asyncio.Queue provides a thread-unsafe but coroutine-safe queue for async workflows.",
    "Process pools in Python use pickle to serialise arguments, which can be a bottleneck.",
    "Dask provides parallel computing for Python with lazy evaluation and task graph scheduling.",
    "Numba compiles Python functions to native machine code using LLVM, bypassing the GIL.",
    "The multiprocessing.Manager class enables shared state between processes via a proxy server.",
]


async def main():
    llm = OpenAILLM(model="gpt-4o-mini")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = InMemoryVectorStore(embeddings=embeddings)

    await vectorstore.aadd(DOCS)

    base_retriever = VectorRetriever(vectorstore=vectorstore, top_k=20)
    reranker = CrossEncoderReranker(
        model="cross-encoder/ms-marco-MiniLM-L-6-v2",
        top_k=5,
    )

    rag = RAG(llm=llm, retriever=base_retriever, reranker=reranker)

    questions = [
        "What is the best way to run CPU-bound Python code in parallel?",
        "How does asyncio handle concurrency without threads?",
        "What is the difference between asyncio.Queue and queue.Queue?",
    ]

    for question in questions:
        print(f"Q: {question}")
        result = await rag.aquery(question)
        print(f"A: {result.answer}\n")


asyncio.run(main())

## Next Steps

- [RAG Fusion](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/rag-fusion) — generate a richer candidate pool before reranking using multi-query fusion
- [Self-RAG](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/self-rag) — after reranking, grade the top-k for hallucination before generation
- [Parent Document Retriever](https://synapsekit.github.io/synapsekit-docs/guides/retrieval/parent-document-retriever) — rerank parent chunks (full-context documents) rather than small child chunks